In [3]:
import glob, pandas as pd, numpy as np
from pathlib import Path

ROOT = next(folder for folder in (Path.cwd(), Path.cwd().parent) if (folder / "data").is_dir())
DATA = ROOT / "data" / "cleaned"
pd.set_option("display.width", 200); pd.set_option("display.float_format","{:,.2f}".format)

spend  = pd.read_csv(DATA / "marketing_spend.csv", parse_dates=["date"])
users  = pd.read_csv(DATA / "users.csv", parse_dates=["signup_ts"])

Q3_START, Q3_END = pd.Timestamp("2026-07-01"), pd.Timestamp("2026-10-01")

# ── A. Schema ───────────────────────────────────────────────────────
print("spend columns:", list(spend.columns))
print("users columns:", list(users.columns))


spend columns: ['date', 'channel', 'spend_egp', 'impressions', 'clicks', 'platform_reported_installs']
users columns: ['user_id', 'signup_ts', 'full_name', 'governorate', 'utm_source', 'utm_campaign', 'kyc_verified_ts', 'utm_source_normalized', 'utm_campaign_normalized']


In [4]:

# ── B. Date-range compliance ───────────────────────────────────────
print(f"\nspend.date: {spend.date.min().date()} → {spend.date.max().date()}")
print(f"rows total: {len(spend)}   |   rows in Q3: {len(spend[(spend.date>=Q3_START)&(spend.date<Q3_END)])}")
print("Rows OUTSIDE Q3 (must exclude from the CPI answer):")
print(spend[(spend.date<Q3_START)|(spend.date>=Q3_END)].head(10).to_string(index=False))




spend.date: 2026-07-01 → 2026-09-30
rows total: 276   |   rows in Q3: 276
Rows OUTSIDE Q3 (must exclude from the CPI answer):
Empty DataFrame
Columns: [date, channel, spend_egp, impressions, clicks, platform_reported_installs]
Index: []


In [5]:
# ── C. DQ on marketing_spend ──────────────────────────────────────
q3 = spend[(spend.date>=Q3_START)&(spend.date<Q3_END)].copy()
print("\nChannels:", sorted(q3.channel.unique().tolist()))
print("Days present per channel (expect 92 each):")
print(q3.groupby("channel").date.nunique().to_string())

print("\nDuplicate (date,channel):", q3.duplicated(["date","channel"]).sum())
print("Negative/zero spend     :", (q3.spend_egp<=0).sum())
print("Spend>0, installs=0     :", ((q3.spend_egp>0)&(q3.platform_reported_installs==0)).sum())
print("Installs>0, spend=0     :", ((q3.spend_egp==0)&(q3.platform_reported_installs>0)).sum())
print("Clicks>0?               :", (q3.clicks>0).all(), "| rows with clicks=0:", (q3.clicks==0).sum())




Channels: ['Google Ads', 'Meta', 'TikTok']
Days present per channel (expect 92 each):
channel
Google Ads    92
Meta          92
TikTok        92

Duplicate (date,channel): 0
Negative/zero spend     : 0
Spend>0, installs=0     : 0
Installs>0, spend=0     : 0
Clicks>0?               : True | rows with clicks=0: 0


In [6]:
# ── D. CPI (spend-weighted) — the headline ────────────────────────
cpi = (q3.groupby("channel")
       .agg(spend_egp=("spend_egp","sum"),
            reported_installs=("platform_reported_installs","sum"),
            clicks=("clicks","sum"), impressions=("impressions","sum"))
       .assign(cpi_reported=lambda d: d.spend_egp/d.reported_installs,
               cpc        =lambda d: d.spend_egp/d.clicks,
               cpm        =lambda d: 1000*d.spend_egp/d.impressions,
               ctr_pct    =lambda d: 100*d.clicks/d.impressions,
               ctir_pct   =lambda d: 100*d.reported_installs/d.clicks)
       .sort_values("cpi_reported"))
print("\n── Q3 CPI (spend-weighted) ──")
print(cpi[["spend_egp","reported_installs","cpi_reported","cpc","cpm","ctr_pct","ctir_pct"]].round(2))

# ── E. Reported installs vs ACTUAL signups (the check that matters) ─
MAP = {"meta":"Meta","facebook ads":"Meta","facebook":"Meta","fb":"Meta","instagram":"Meta","ig":"Meta",
       "google":"Google Ads","google ads":"Google Ads","adwords":"Google Ads","gads":"Google Ads",
       "tiktok":"TikTok","tt":"TikTok"}
def map_ch(raw):
    if pd.isna(raw) or str(raw).strip() in ("","(direct)"): return "Unknown"
    return MAP.get(str(raw).strip().lower(), "Other")

users["ch"] = users.utm_source.map(map_ch)
u_q3 = users[(users.signup_ts>=Q3_START)&(users.signup_ts<Q3_END)
             & (users.full_name.str.lower()!="test qa")]

actual = u_q3.groupby("ch").size().rename("actual_signups")
compare = (cpi[["spend_egp","reported_installs"]]
           .join(actual, how="inner")
           .assign(inflation=lambda d: d.reported_installs/d.actual_signups,
                   cpi_true =lambda d: d.spend_egp/d.actual_signups))
print("\n── Reported vs Actual ──")
print(compare.round(2))


── Q3 CPI (spend-weighted) ──
            spend_egp  reported_installs  cpi_reported   cpc      cpm  ctr_pct  ctir_pct
channel                                                                                 
TikTok      42,652.36               1185         35.99 11.81    94.49     0.80     32.82
Meta       145,369.33               3355         43.33 14.37   158.10     1.10     33.17
Google Ads 168,517.02               1803         93.46 29.18 1,021.65     3.50     31.22

── Reported vs Actual ──
            spend_egp  reported_installs  actual_signups  inflation  cpi_true
channel                                                                      
TikTok      42,652.36               1185             650       1.82     65.62
Meta       145,369.33               3355            2100       1.60     69.22
Google Ads 168,517.02               1803            1430       1.26    117.84


In [11]:
import glob, pandas as pd
from pathlib import Path

raw_users = pd.read_csv(next((ROOT / "data" / "raw").glob("* - users.csv")),
                    parse_dates=["signup_ts","kyc_verified_ts"])
Q3_S, Q3_E, EXTRACT = (pd.Timestamp("2026-07-01"),
                       pd.Timestamp("2026-10-01"),
                       pd.Timestamp("2026-11-05"))

# ── DQ on raw_users BEFORE using it as denominator ─────────────────────
print("Rows total:", len(raw_users))

# 1. user_id uniqueness
print("Duplicate user_id:", raw_users.user_id.duplicated().sum())

# 2. Future signups (after extract date)
print("signup_ts > extract date:", (raw_users.signup_ts > EXTRACT).sum())

# 3. KYC before signup (physically impossible)
print("kyc_verified_ts < signup_ts:", (raw_users.kyc_verified_ts < raw_users.signup_ts).sum())

print()

# 4. Test / QA accounts
qa = raw_users[raw_users.full_name.str.lower().str.contains("test|qa", na=False)]
print(qa[["user_id","full_name","signup_ts","kyc_verified_ts"]])

print()

# 5. Null/blank utm_source rate overall AND in Q3
q3 = raw_users[(raw_users.signup_ts>=Q3_S)&(raw_users.signup_ts<Q3_E)]
null_utm_all = raw_users.utm_source.isna().sum() + (raw_users.utm_source.fillna("").str.strip()=="").sum()
null_utm_q3  = q3.utm_source.isna().sum()  + (q3.utm_source.fillna("").str.strip()=="").sum()
print(f"Blank/NA utm_source — all time: {null_utm_all}/{len(raw_users)}  ({100*null_utm_all/len(raw_users):.1f}%)")
print(f"Blank/NA utm_source — Q3      : {null_utm_q3}/{len(q3)}    ({100*null_utm_q3/len(q3):.1f}%)")

# 6. (direct) counts — treated separately in my mapping?
print("(direct) rows:", (raw_users.utm_source.fillna("").str.strip()=="(direct)").sum())

# 7. utm_source spelling variants (did my MAP dict cover everything?)
print("\nRaw utm_source values in Q3:")
print(q3.utm_source.fillna("<blank>").value_counts().to_string())

Rows total: 6230
Duplicate user_id: 0
signup_ts > extract date: 0
kyc_verified_ts < signup_ts: 0

      user_id  full_name           signup_ts     kyc_verified_ts
649    106224    test qa 2026-09-21 05:28:52 2026-09-21 05:33:52
688    106228    test qa 2026-08-25 06:20:21 2026-08-25 06:25:21
713    106230  test test 2026-08-17 10:41:52 2026-08-17 10:46:52
777    106221  Test User 2026-07-01 02:56:43 2026-07-01 03:01:43
796    106207    test qa 2026-08-01 15:22:50 2026-08-01 15:27:50
1520   106211    test qa 2026-08-24 01:30:13 2026-08-24 01:35:13
1726   106208  Test User 2026-07-28 01:18:57 2026-07-28 01:23:57
2295   106222  test test 2026-09-28 12:07:08 2026-09-28 12:12:08
2692   106225   QA Lucky 2026-09-13 22:45:28 2026-09-13 22:50:28
2808   106226   QA Lucky 2026-09-07 21:59:45 2026-09-07 22:04:45
2928   106219    test qa 2026-08-13 22:45:21 2026-08-13 22:50:21
3076   106206   QA Lucky 2026-09-24 03:52:54 2026-09-24 03:57:54
3399   106209   QA Lucky 2026-09-25 09:18:40 2026-09-25 0

In [12]:
# Users have already been cleaned in Task 2.
u_q3 = users[(users.signup_ts >= Q3_S) & (users.signup_ts < Q3_E)].copy()
u_q3["ch"] = u_q3.utm_source.map(map_ch)

# Rebuild
actual = u_q3.groupby("ch").size().rename("actual_signups")
final = (cpi[["spend_egp","reported_installs"]]
         .join(actual, how="inner")
         .assign(cost_per_signup=lambda d: d.spend_egp/d.actual_signups))
print(final.round(2))
print("\nUnattributed share:",
      f"{100*(1 - final.actual_signups.sum()/len(u_q3)):.1f}%")

            spend_egp  reported_installs  actual_signups  cost_per_signup
channel                                                                  
TikTok      42,652.36               1185             650            65.62
Meta       145,369.33               3355            2100            69.22
Google Ads 168,517.02               1803            1430           117.84

Unattributed share: 32.6%


In [13]:
# Users have already been cleaned in Task 2.
u_q3 = users[(users.signup_ts >= Q3_S) & (users.signup_ts < Q3_E)].copy()
u_q3["ch"] = u_q3.utm_source.map(map_ch)

print("\nBucket counts (verify 'Other' = 0):")
print(u_q3.ch.value_counts().to_string())

actual = u_q3.groupby("ch").size().rename("actual_signups")

final = (cpi[["spend_egp","reported_installs"]]
         .join(actual, how="inner")
         .assign(inflation      = lambda d: d.reported_installs / d.actual_signups,
                 cost_per_signup = lambda d: d.spend_egp / d.actual_signups))
print("\n── Final ──")
print(final.round(2))


Bucket counts (verify 'Other' = 0):
ch
Meta          2100
Google Ads    1430
Unknown       1015
Other         1005
TikTok         650

── Final ──
            spend_egp  reported_installs  actual_signups  inflation  cost_per_signup
channel                                                                             
TikTok      42,652.36               1185             650       1.82            65.62
Meta       145,369.33               3355            2100       1.60            69.22
Google Ads 168,517.02               1803            1430       1.26           117.84


NameError: name 'DATA_DIR' is not defined